In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
# ============================================================
# SAMSUNG MOBILE RECOMMENDATION SYSTEM
# Content-Based Filtering + Collaborative Filtering
# Dataset: samsung_mobile_recommendation_50_samples.csv
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# -------------------- 1. LOAD DATASET -------------------------
# Google Colab: upload the CSV when prompted.
from google.colab import files
import pandas as pd

uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload the CSV file.")

csv_file = next(iter(uploaded.keys()))
df = pd.read_csv(csv_file)

print("Dataset uploaded successfully!")
print("Total samples:", len(df))
display(df.head())

 # For VS Code, keep the CSV in the same folder as this script.
csv_file = "samsung_mobile_recommendation_50_samples.csv"

df = pd.read_csv(csv_file)

required_columns = [
    "User_ID", "Samsung_Model", "Approx_Budget_INR", "Primary_Use",
    "Camera_Preference_Score", "Battery_Preference_Score",
    "Performance_Preference_Score", "User_Rating_1_to_5"
]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing columns: {missing}")

print("\nDataset loaded successfully!")
print("Rows:", len(df))
print(df.head())

# -------------------- 2. MOBILE CATALOG -----------------------
# Representative demonstration values, not verified live prices/specifications.
# One row per model is used for content-based matching.
catalog = (
    df.groupby("Samsung_Model", as_index=False)
      .agg(
          Approx_Budget_INR=("Approx_Budget_INR", "median"),
          Camera_Score=("Camera_Preference_Score", "median"),
          Battery_Score=("Battery_Preference_Score", "median"),
          Performance_Score=("Performance_Preference_Score", "median")
      )
)

# Feature columns used to compare the user's preferences to each phone.
feature_cols = ["Camera_Score", "Battery_Score", "Performance_Score"]
scaler = MinMaxScaler()
catalog_scaled = scaler.fit_transform(catalog[feature_cols])

# -------------------- 3. USER INPUT ---------------------------
print("\n--- Tell us what you need ---")
budget = float(input("Your maximum budget in INR (e.g. 30000): "))
print("Primary use options: Study, Camera, Gaming, Battery, Photography, Social Media, All-rounder, Budget")
primary_use = input("Your primary use: ").strip().title()

camera_pref = int(input("Camera importance (1=low, 5=high): "))
battery_pref = int(input("Battery importance (1=low, 5=high): "))
performance_pref = int(input("Performance/gaming importance (1=low, 5=high): "))

for label, value in [
    ("Camera", camera_pref),
    ("Battery", battery_pref),
    ("Performance", performance_pref)
]:
    if not 1 <= value <= 5:
        raise ValueError(f"{label} importance must be between 1 and 5.")

# -------------------- 4. CONTENT-BASED FILTERING --------------
# Compare the user's importance weights with each model's feature scores.
user_weights = np.array([camera_pref, battery_pref, performance_pref], dtype=float)
user_weights = user_weights / user_weights.sum()

# Normalize feature scores and compute a weighted preference score.
content_scores = catalog_scaled @ user_weights
catalog["Content_Score"] = content_scores

# Budget fit: phones over budget are excluded; phones within budget are ranked.
affordable = catalog[catalog["Approx_Budget_INR"] <= budget].copy()

if affordable.empty:
    print("\nNo model in this sample dataset fits your budget.")
    print("Try increasing your budget. No phone recommendation can be made from this catalogue.")
    raise SystemExit

# Optional primary-use adjustment based on the demo data's feature patterns.
# It is deliberately small so the explicit camera/battery/performance weights remain important.

use_bonus = {
    "Camera": "Camera_Score",
    "Photography": "Camera_Score",
    "Battery": "Battery_Score",
    "Gaming": "Performance_Score",
    "Study": None,
    "Social Media": "Camera_Score",
    "All-rounder": None,
    "Budget": None
}

preferred_feature = use_bonus.get(primary_use)

if preferred_feature is not None:
    affordable["Content_Score"] += 0.05 * (
        affordable[preferred_feature] / 100.0
    )

print("Primary-use adjustment applied successfully!")



Saving samsung_mobile_recommendation_50_samples.csv to samsung_mobile_recommendation_50_samples (2).csv
Dataset uploaded successfully!
Total samples: 50


,User_ID,Samsung_Model,Approx_Budget_INR,Primary_Use,Camera_Preference_Score,Battery_Preference_Score,Performance_Preference_Score,User_Rating_1_to_5
0,U01,Galaxy A15 5G,18000,Study,30,25,25,4
1,U02,Galaxy A25 5G,25000,Camera,45,30,25,4
2,U03,Galaxy A35 5G,30000,All-rounder,40,35,35,5
3,U04,Galaxy A15 5G,18000,Battery,25,30,45,4
4,U05,Galaxy S24 FE,60000,Gaming,35,30,50,5



Dataset loaded successfully!
Rows: 50
  User_ID  Samsung_Model  Approx_Budget_INR  Primary_Use  \
0     U01  Galaxy A15 5G              18000        Study   
1     U02  Galaxy A25 5G              25000       Camera   
2     U03  Galaxy A35 5G              30000  All-rounder   
3     U04  Galaxy A15 5G              18000      Battery   
4     U05  Galaxy S24 FE              60000       Gaming   

   Camera_Preference_Score  Battery_Preference_Score  \
0                       30                        25   
1                       45                        30   
2                       40                        35   
3                       25                        30   
4                       35                        30   

   Performance_Preference_Score  User_Rating_1_to_5  
0                            25                   4  
1                            25                   4  
2                            35                   5  
3                            45                